Creating input data from observations

In [1]:
# Deterministic hash ordering for set()/dict iteration across runs
import os
os.environ["PYTHONHASHSEED"] = "0"

import pandas as pd
import numpy as np

observations = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl'))
pos_stars = sorted(set(observations[observations['has_exoplanets'] == 1]['star_name']))
neg_stars = sorted(set(observations[observations['has_exoplanets'] == 0]['star_name']))

# Sinusoidal positional encoding for BJD
NUM_FREQS = 8
min_period = 1.0
max_period = 7300.0
periods = np.logspace(np.log10(min_period), np.log10(max_period), NUM_FREQS)
freqs = 2.0 * np.pi / periods

def bjd_positional_encoding(bjd, ref_bjd):
    dt = bjd - ref_bjd
    encoding = []
    for f in freqs:
        encoding.append(np.sin(f * dt))
        encoding.append(np.cos(f * dt))
    return encoding

pos_inputs = []
neg_inputs = []

for star in pos_stars:
    star_obs = observations[observations['star_name'] == star].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]

    rows = []
    for idx in range(len(star_obs)):
        row = list(star_obs.iloc[idx])
        # Columns at this point: star_name, bjd, rv, rv_err, exposure_time, RHKp, Halpha, has_exoplanets, rv_centered
        # Indices:              0          1    2   3       4              5      6       7               8
        bjd = row[1]
        rv_centered = row[8]
        rv_err = row[3]
        exposure_time = row[4]
        rhkp = row[5]
        halpha = row[6]

        pos_enc = bjd_positional_encoding(bjd, ref_bjd)

        features = [rv_centered, rv_err, exposure_time, rhkp, halpha] + pos_enc
        rows.append(features)

    pos_inputs.append(np.array(rows))

for star in neg_stars:
    star_obs = observations[observations['star_name'] == star].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]

    rows = []
    for idx in range(len(star_obs)):
        row = list(star_obs.iloc[idx])
        bjd = row[1]
        rv_centered = row[8]
        rv_err = row[3]
        exposure_time = row[4]
        rhkp = row[5]
        halpha = row[6]

        pos_enc = bjd_positional_encoding(bjd, ref_bjd)

        features = [rv_centered, rv_err, exposure_time, rhkp, halpha] + pos_enc
        rows.append(features)

    neg_inputs.append(np.array(rows))

print(f"Positive stars: {len(pos_inputs)}, total observations: {sum(len(x) for x in pos_inputs)}")
print(f"Negative stars: {len(neg_inputs)}, total observations: {sum(len(x) for x in neg_inputs)}")
print(f"Feature dimension per observation: {pos_inputs[0].shape[1]}")

Positive stars: 413, total observations: 46533
Negative stars: 1774, total observations: 189034
Feature dimension per observation: 21


Creating Data Splits

Creating Model

Training model

Model Evaluation on Test Set

In [ ]:
# OOF protocol: build star_name -> sequence + label maps
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold

def seed_everything(seed):
    import random, torch, numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Build star_name -> sequence + label maps
star_to_seq = {}

# Build name->seq maps from cell 1 data
pos_by_name = {s: seq for s, seq in zip(pos_stars, pos_inputs)}
neg_by_name = {s: seq for s, seq in zip(neg_stars, neg_inputs)}

star_to_label = {}
for star in pos_stars:
    star_to_seq[star] = pos_by_name[star]
    star_to_label[star] = 1
for star in neg_stars:
    star_to_seq[star] = neg_by_name[star]
    star_to_label[star] = 0

all_stars = sorted(star_to_seq.keys())
y = np.array([star_to_label[s] for s in all_stars], dtype=int)
all_seqs = [star_to_seq[s] for s in all_stars]
n_stars = len(all_stars)
print(f"Stars: {n_stars}, pos={y.sum()}, neg={(1-y).sum()}")

# Truncation length (same as original)
MAX_SEQ_LEN = 100

def truncate_star(s, max_len=MAX_SEQ_LEN):
    if len(s) > max_len:
        return s[:max_len]
    return s

# Pre-truncate all sequences
all_seqs = [truncate_star(s) for s in all_seqs]
lens = [len(s) for s in all_seqs]
print(f"Sequence lengths after truncation: min={min(lens)}, max={max(lens)}, median={int(np.median(lens))}")



In [ ]:
# Dataset + collate (same as original)
class StarDataset(Dataset):
    def __init__(self, seqs, labels):
        self.data = seqs
        self.labels = labels
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        star = torch.tensor(self.data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return star, label

def collate_stars(batch):
    stars, labels = zip(*batch)
    max_len = max(s.shape[0] for s in stars)
    padded = []
    mask = []
    for star in stars:
        seq_len = star.shape[0]
        pad_len = max_len - seq_len
        feat_dim = star.shape[1]
        if pad_len > 0:
            padding = torch.zeros(pad_len, feat_dim)
            padded_star = torch.cat([star, padding], dim=0)
        else:
            padded_star = star
        star_mask = torch.cat([torch.ones(seq_len), torch.zeros(pad_len)])
        padded.append(padded_star)
        mask.append(star_mask)
    padded = torch.stack(padded)
    mask = torch.stack(mask)
    labels = torch.stack(labels)
    return padded, mask, labels


In [ ]:
# Model definition (same architecture)
import torch.nn as nn
import torch.nn.functional as F

class AttentionPool(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = nn.Linear(d_model, 1)
    def forward(self, x, mask):
        scores = self.attention(x).squeeze(-1)
        scores = scores.masked_fill(~mask.bool(), float('-inf'))
        weights = F.softmax(scores, dim=1)
        pooled = (x * weights.unsqueeze(-1)).sum(dim=1)
        return pooled

class ExoplanetTransformer(nn.Module):
    def __init__(self, feat_dim=21, d_model=48, nhead=4, num_layers=1, dim_feedforward=96, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = AttentionPool(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1),
        )
    def forward(self, x, mask):
        x = self.input_proj(x)
        src_key_padding_mask = ~mask.bool()
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        x = self.pool(x, mask)
        out = self.classifier(x).squeeze(-1)
        return out

import math
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

# Training config (no val-based early stopping in OOF)
N_EPOCHS = 100
BATCH = 32
LR = 1e-3
WD = 5e-3
WARMUP = 5

def train_one_fold(train_idx, test_idx, rep_seed):
    seed_everything(rep_seed)
    train_seqs = [all_seqs[i] for i in train_idx]
    train_labels_list = [float(y[i]) for i in train_idx]
    test_seqs = [all_seqs[i] for i in test_idx]

    # Standardize using training stats
    train_cat = np.concatenate(train_seqs, axis=0)
    feat_mean = train_cat.mean(axis=0)
    feat_std = train_cat.std(axis=0)
    feat_std = np.clip(feat_std, 1e-8, None)
    train_seqs = [(s - feat_mean) / feat_std for s in train_seqs]
    test_seqs = [(s - feat_mean) / feat_std for s in test_seqs]

    ds_tr = StarDataset(train_seqs, train_labels_list)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, drop_last=False,
                       collate_fn=collate_stars, pin_memory=(device.type=='cuda'))

    model = ExoplanetTransformer().to(device)
    n_pos_train = sum(train_labels_list)
    n_neg_train = len(train_labels_list) - n_pos_train
    pos_w = torch.tensor([math.sqrt(n_neg_train / max(n_pos_train, 1))], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = Adam(model.parameters(), lr=LR, weight_decay=WD)

    def lr_fn(ep):
        if ep < WARMUP:
            return (ep + 1) / WARMUP
        progress = (ep - WARMUP) / (N_EPOCHS - WARMUP)
        return 0.5 * (1 + math.cos(math.pi * progress))
    sched = LambdaLR(opt, lr_fn)

    model.train()
    for ep in range(N_EPOCHS):
        for xb, mask_b, yb in dl_tr:
            xb, mask_b, yb = xb.to(device), mask_b.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb, mask_b), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        sched.step()

    model.eval()
    ds_te = StarDataset(test_seqs, [0.0]*len(test_seqs))
    dl_te = DataLoader(ds_te, batch_size=BATCH, shuffle=False,
                       collate_fn=collate_stars, pin_memory=(device.type=='cuda'))
    all_logits = []
    with torch.no_grad():
        for xb, mask_b, _ in dl_te:
            xb, mask_b = xb.to(device), mask_b.to(device)
            logits = model(xb, mask_b).cpu().numpy()
            all_logits.append(logits)
    logits = np.concatenate(all_logits)
    logits = np.nan_to_num(logits, nan=0.0, posinf=35.0, neginf=-35.0)
    probs = 1.0 / (1.0 + np.exp(-logits))
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0).astype(np.float32)
    return probs

print("Model and train_one_fold defined.")


In [ ]:
# OOF Evaluation: 10-fold StratifiedKFold × 5 reps
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)
from sklearn.model_selection import StratifiedKFold
import pandas as pd

N_REPS = 5
N_FOLDS = 10

print(f"{"="*72}")
print(f"OUT-OF-FOLD EVALUATION — Transformer")
print(f"{"="*72}")
print(f"Protocol: {N_FOLDS}-fold StratifiedKFold × {N_REPS} reps, seeds 42-{41+N_REPS}")
print(f"Epochs: {N_EPOCHS}, Batch: {BATCH}, LR: {LR}, WD: {WD}")
print()

all_oof_probs = np.zeros((n_stars, N_REPS))
rep_metrics = {'rep': [], 'pr_auc': [], 'roc_auc': [],
               'f1': [], 'f05': [], 'precision': [], 'recall': []}

for rep in range(N_REPS):
    oof_probs = np.zeros(n_stars)
    rep_seed = 42 + rep
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep_seed)
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(np.zeros(n_stars), y)):
        probs = train_one_fold(train_idx, test_idx, rep_seed)
        oof_probs[test_idx] = probs
        if (fold_idx + 1) % 2 == 0:
            print(f"  rep {rep} fold {fold_idx+1}/{N_FOLDS} done")
    all_oof_probs[:, rep] = oof_probs

    roc = roc_auc_score(y, oof_probs)
    pr  = average_precision_score(y, oof_probs)
    vp, vr, vt = precision_recall_curve(y, oof_probs)
    vf1 = 2 * vp * vr / (vp + vr + 1e-8)
    bi = int(np.argmax(vf1))
    thr = float(vt[bi]) if bi < len(vt) else 0.5
    preds = (oof_probs >= thr).astype(int)
    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp+fp) if (tp+fp)>0 else 0.0
    rec = tp / (tp+fn) if (tp+fn)>0 else 0.0
    f1  = f1_score(y, preds, zero_division=0)
    f05 = fbeta_score(y, preds, beta=0.5, zero_division=0)
    rep_metrics['rep'].append(rep); rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc); rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc); rep_metrics['recall'].append(rec)
    print(f"  Rep {rep} seed={rep_seed}: PR-AUC={pr:.4f}  ROC={roc:.4f}  F1={f1:.4f}  F0.5={f05:.4f}  P={prc:.3f}  R={rec:.3f}")

# Summary
rep_df = pd.DataFrame(rep_metrics)
print(f"\n{"="*72}")
print(f"Transformer OOF SUMMARY: 10-fold × 5 reps")
print(f"{"="*72}")
print("\nAggregate (n=5 reps):")
for m in ['pr_auc','roc_auc','f1','f05','precision','recall']:
    v = rep_df[m].values
    print(f"  {m:11s}: {v.mean():.4f} +/- {v.std(ddof=1):.4f}  (min={v.min():.4f}, max={v.max():.4f})")

# Combined OOF
avg_oof = all_oof_probs.mean(axis=1)
combined_pr  = average_precision_score(y, avg_oof)
combined_roc = roc_auc_score(y, avg_oof)
vp, vr, vt = precision_recall_curve(y, avg_oof)
vf1 = 2 * vp * vr / (vp + vr + 1e-8)
bi = int(np.argmax(vf1))
thr = float(vt[bi]) if bi < len(vt) else 0.5
combined_preds = (avg_oof >= thr).astype(int)
combined_f1   = f1_score(y, combined_preds, zero_division=0)
combined_f05  = fbeta_score(y, combined_preds, beta=0.5, zero_division=0)
cm = confusion_matrix(y, combined_preds)
tn, fp, fn, tp = cm.ravel()
print(f"\n{"="*72}")
print("COMBINED OOF")
print(f"{"="*72}")
print(f"  PR-AUC:  {combined_pr:.4f}")
print(f"  ROC-AUC: {combined_roc:.4f}")
print(f"  F1:      {combined_f1:.4f}  F0.5={combined_f05:.4f}  (threshold={thr:.4f})")
print(f"  P={tp/(tp+fp) if (tp+fp)>0 else 0:.3f}  R={tp/(tp+fn) if (tp+fn)>0 else 0:.3f}  (TN={tn} FP={fp} FN={fn} TP={tp})")

# Bootstrap CIs
from split import bootstrap_roc_auc, bootstrap_pr_auc
pr_point, pr_lo, pr_hi = bootstrap_pr_auc(y, avg_oof)
roc_point, roc_lo, roc_hi = bootstrap_roc_auc(y, avg_oof)
print(f"\nBOOTSTRAP 95% CI (200 resamples):")
print(f"  PR-AUC:  {pr_point:.4f}  [{pr_lo:.4f}, {pr_hi:.4f}]")
print(f"  ROC-AUC: {roc_point:.4f}  [{roc_lo:.4f}, {roc_hi:.4f}]")

# Save results
import pickle
results = {
    'model': 'Transformer',
    'pr_auc_combined': combined_pr,
    'roc_auc_combined': combined_roc,
    'f1': combined_f1, 'f05': combined_f05,
    'pr_auc_bootstrap': (pr_point, pr_lo, pr_hi),
    'roc_auc_bootstrap': (roc_point, roc_lo, roc_hi),
    'rep_metrics': rep_df.to_dict('records'),
    'n_stars': n_stars,
}
with open('/kaggle/working/transformer_oof_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to transformer_oof_results.pkl")
print(f"FINAL: Transformer combined OOF PR-AUC = {combined_pr:.4f}")


# Transformer OOF Evaluation

**OOF protocol:** 10-fold StratifiedKFold x 5 reps (seeds 42-46).
Every star gets a prediction from a Transformer that never saw it.
